# E-commerce A/B Test — Cleaning

## Goal

Produce an analysis-ready dataset by handling the data quality issues found:
- **3,893 mismatched rows** — users assigned to one group but shown the wrong page
- **3,894 duplicate users** — users appearing more than once

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

df = pd.read_csv("/Users/edi/Documents/ab-test-analysis/data/ab_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Starting with: {len(df):,} rows")

Starting with: 294,478 rows


## Step 1: Remove mismatched rows

These are users where `group` and `landing_page` disagree. We cannot include them in the analysis because we cannot determine what they were actually "exposed to" — from their experience perspective, they got a treatment version when assigned to control, or vice versa. Including them would contaminate both groups.

In [2]:
# Identify mismatched rows
mismatch_mask = (
    ((df["group"] == "control") & (df["landing_page"] != "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] != "new_page"))
)

before = len(df)
df = df[~mismatch_mask].copy()
dropped = before - len(df)

print(f"Dropped {dropped:,} mismatched rows")
print(f"Remaining: {len(df):,}")

Dropped 3,893 mismatched rows
Remaining: 290,585


## Step 2: Handle duplicate users

Some users appear more than once. Statistically, an A/B test assumes each user is one independent observation. Duplicates violate this assumption and also suggest the user's experience may not have been fully consistent (they may have been re-assigned on a subsequent visit).

**Decision:** keep only the first visit per user. This reflects the user's initial exposure to their assigned variant.

In [3]:
# Check for remaining duplicates after mismatch cleanup
user_counts = df["user_id"].value_counts()
dupes = user_counts[user_counts > 1]
print(f"Remaining duplicate users after mismatch cleanup: {len(dupes):,}")

Remaining duplicate users after mismatch cleanup: 1


In [4]:
# Sort by timestamp, keep first occurrence of each user
df = df.sort_values(["user_id", "timestamp"]).drop_duplicates(
    subset="user_id", keep="first"
).copy()

print(f"Rows after deduplication: {len(df):,}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"(Should match — one row per user)")

Rows after deduplication: 290,584
Unique users: 290,584
(Should match — one row per user)


## Step 3: check the cleaned dataset

Verify the cleaned data still has a healthy group balance (randomization preserved through cleaning) and compute the cleaned conversion rates.

In [6]:
# Group balance check after cleaning
group_counts = df.groupby("group").size()
total = len(df)
print("Group sizes after cleaning:")
print(group_counts)
print(f"\nControl share:   {group_counts['control']/total*100:.3f}%")
print(f"Treatment share: {group_counts['treatment']/total*100:.3f}%")


Group sizes after cleaning:
group
control      145274
treatment    145310
dtype: int64

Control share:   49.994%
Treatment share: 50.006%


In [7]:
# Clean conversion rates
conv_by_group = df.groupby("group")["converted"].agg(["count", "sum", "mean"])
conv_by_group.columns = ["n_users", "n_converted", "conversion_rate"]
conv_by_group["conversion_pct"] = (conv_by_group["conversion_rate"] * 100).round(3)
print("Cleaned conversion rates by group:")
conv_by_group


Cleaned conversion rates by group:


,n_users,n_converted,conversion_rate,conversion_pct
group,,,,
control,145274,17489,0.1204,12.0390
treatment,145310,17264,0.1188,11.8810


In [8]:
df.to_csv("/Users/edi/Documents/ab-test-analysis/data/ab_data_clean.csv", index=False)
print(f"Saved cleaned dataset: {len(df):,} rows")

Saved cleaned dataset: 290,584 rows
